In [19]:
import os
import numpy as np

# Set the path to the directory containing .npy files
trajectory_dir = '/Users/vaibhavmishra/Desktop/Desktop/btx-game-aicode/clash_squad_agent_trajectories'

# Get list of all .npy files in the directory
npy_files = [f for f in os.listdir(trajectory_dir) if f.endswith('.npy')]


def prepare_features(data):
    chunk_size = 20
    i=0
    features = []
    while True:
        if i + chunk_size >= len(data):
            break
        feature = data[i:i+chunk_size]
        diff = data[i+chunk_size -1 :i+chunk_size+1]
        delta_x = diff[1][0] - diff[0][0]
        delta_z = diff[1][1] - diff[0][1]
        delta_rotation = diff[1][2] - diff[0][2]
        action = [delta_x, delta_z, delta_rotation]
        features.append({"feature": feature, "action": action})
        i += 1
    return features




In [ ]:
new_output_dir = "/Users/vaibhavmishra/Desktop/Desktop/btx-game-aicode/clash_squad_agent_trajectories_processed"
action_output_dir = os.path.join(new_output_dir, "actions")
feature_output_dir = os.path.join(new_output_dir, "features")
os.makedirs(new_output_dir, exist_ok=True)
os.makedirs(action_output_dir, exist_ok=True)
os.makedirs(feature_output_dir, exist_ok=True)
def save_features(features, filename_prefix, count):

    # Extract features and actions separately
    for item in features:
        feature_array = item['feature']
        action_array = item['action']
        if(action_array[0]>0.5):
            action_array[0] = 0.5
        elif(action_array[0] < -0.5):
            action_array[0] = -0.5
        if(action_array[1]>0.5):
            action_array[1] = 0.5
        elif(action_array[1] < -0.5):
            action_array[1] = -0.5
        
        feature_array[:, 0] = feature_array[:, 0] / 500.0
        feature_array[:, 1] = feature_array[:, 1] / 500.0 
        feature_array[:, 2] = feature_array[:, 2] / 360
        feature_array[:, 5] = feature_array[:, 5] / 100
        feature_array[:, 6] = 0
        feature_array[:, 7] = feature_array[:, 7] / 500.0 
        feature_array[:, 8] = feature_array[:, 8] / 500.0 
        feature_array[:, 9] = feature_array[:, 9] / 360
        feature_array[:, 16] = feature_array[:, 16] / 500.0 
        

        features_np = np.array(feature_array)  # Shape: (n_samples, 20, 18)
        actions_np = np.array(action_array)    # Shape: (n_samples, 3)
        
        features_path = os.path.join(feature_output_dir, f"{filename_prefix}_features_{count}.npy")
        actions_path = os.path.join(action_output_dir, f"{filename_prefix}_actions_{count}.npy")
        
        np.save(features_path, features_np)
        np.save(actions_path, actions_np)
        count +=1
        
        
    return count


In [21]:
# Example usage of the save_features function

# Process all files and save them
print("Processing all trajectory files...")
count = 0
for i, npy_file in enumerate(npy_files):
    file_path = os.path.join(trajectory_dir, npy_file)
    data = np.load(file_path)
    features = prepare_features(data)
    
    # Extract filename without extension to use as prefix
    filename_prefix = os.path.splitext(npy_file)[0]
    
    # Save features and actions in separate files
    count = save_features(features, filename_prefix, count)



Processing all trajectory files...
